# Feature Selection:

Now that we have analysed the data, we are ready to choose the features that will be used to train the model.

We first need to choose and preare the group of point-value metrics for the classical ML models.
Then we will need to compute the time-series features derived from the raw data like in 09.2_summarizing_group_of_gait_cycles.ipynb

Once selected, we will analyze the features in detail to choose the ones we will mvoe forward with to the next phase....


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency, pearsonr
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import LabelEncoder

In [ ]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std

def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k - 1)*(r - 1))/(n - 1))
    rcorr = r - ((r - 1)**2)/(n - 1)
    kcorr = k - ((k - 1)**2)/(n - 1)
    return np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))

def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data['feature'] = df.columns
    vif_data['VIF'] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif_data

# ==============================
# Main analysis function
# ==============================

def predictor_analysis(df, target, cat_threshold=10):
    results = []
    
    y = df[target]
    y_num = LabelEncoder().fit_transform(y)  # Binary 0/1 target
    
    for col in df.columns:
        if col == target:
            continue
        
        series = df[col].dropna()
        
        # Detect variable type
        if pd.api.types.is_numeric_dtype(series) and df[col].nunique() > cat_threshold:
            # Continuous variable
            group0 = df[df[target] == y.unique()[0]][col]
            group1 = df[df[target] == y.unique()[1]][col]
            
            # t-test
            t_pval = ttest_ind(group0, group1, equal_var=False).pvalue
            d = cohens_d(group0, group1)
            
            # Pearson
            r_val, r_pval = pearsonr(df[col], y_num)
            
            # Mutual Information
            mi_val = mutual_info_classif(df[[col]], y_num, discrete_features=False)[0]
            
            results.append({
                'predictor': col,
                'type': 'continuous',
                't-test p': t_pval,
                "Cohen's d": d,
                'Pearson r': r_val,
                'Pearson p': r_pval,
                'Mutual Info': mi_val
            })
        
        else:
            # Categorical variable
            contingency = pd.crosstab(df[col], y)
            chi2_pval = chi2_contingency(contingency)[1]
            cramer_v = cramers_v(contingency)
            mi_val = mutual_info_classif(pd.get_dummies(df[[col]]), y_num, discrete_features=True)[0]
            
            results.append({
                'predictor': col,
                'type': 'categorical',
                'Chi2 p': chi2_pval,
                "Cramer's V": cramer_v,
                'Mutual Info': mi_val
            })
    
    results_df = pd.DataFrame(results)
    
    # VIF for continuous predictors
    continuous_cols = [col for col in df.columns if col != target and pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > cat_threshold]
    if continuous_cols:
        vif_df = calculate_vif(df[continuous_cols].fillna(0))
    else:
        vif_df = pd.DataFrame(columns=['feature', 'VIF'])
    
    return results_df, vif_df


In [ ]:
df = pd.read_csv("your_dataset.csv")
results_df, vif_df = predictor_analysis(df, target='your_binary_target')
print(results_df.sort_values('Mutual Info', ascending=False))
print(vif_df)